# SolveBench -- Main Sweep

16 solvers across 930 real sparse matrices from 27 domains.

Refinement is **disabled for every method**, so the comparison carries no wrapper advantage. Success is decided by the harness from the residual it measures itself, never by a solver's own flag.

In [1]:
# pin BLAS threads before numpy import, or timing isn't reproducible on shared hardware
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
print("BLAS threads pinned to 1 for reproducible timing")

BLAS threads pinned to 1 for reproducible timing


In [2]:
# embedded verbatim from src/solvebench/ -- edit the library and regenerate, not here
import sys, pathlib, json

_SOURCES = json.loads(r"""{"config": "# Every tunable number in the benchmark, kept in one place.\n\n# --- iterative solvers -------------------------------------------------------\nMAX_ITERATIONS = 10_000     # cap before a stationary/Krylov method is called non-convergent\nTOLERANCE = 1e-8            # relative residual an iterative method is asked to reach\nSOR_OMEGA = 1.25            # fixed relaxation factor (1.0 would reduce SOR to Gauss-Seidel)\n\nPOWER_ITERS_OMEGA = 60      # power iterations to estimate rho(T_J) before picking omega\n\n# raised from 1e4 -- that value was killing real convergences mid-transient, e.g. cdde6\nDIVERGENCE_GROWTH = 1e12\n\n# --- iterative refinement ----------------------------------------------------\n# orthogonal to the solver -- giving it to one method only is what made APK look best before\nREFINEMENT_PASSES = (0, 1)\nREFINEMENT_STUDY_PASSES = (0, 1, 2)     # third pass barely moves the needle, checked on fs_541_2\n\n# --- ILU preconditioner ------------------------------------------------------\nILU_DROP_TOL = 1e-3\nILU_FILL_FACTOR = 5\n\n# retry ladder, pushes toward robustness each step -- see tools/audit_ilu_hole.py\nILU_LADDER = ((ILU_DROP_TOL, ILU_FILL_FACTOR),\n              (1e-4, 10),\n              (1e-6, 20),\n              (0.0, 50))\n\n# --- direct solvers ----------------------------------------------------------\nDIRECT_SIZE_CAP = 2000      # hand-written, O(n^3) in python, ~35s at n=3000\n\n# --- condition number --------------------------------------------------------\nCOND_EXACT_CAP = 2000       # above this, fall back to a 1-norm estimate\nILL_CONDITIONED = 1e15      # numerically singular past here, reported as its own stratum\n\n# --- spectral analysis -------------------------------------------------------\nSPECTRAL_EXACT_CAP = 2000   # above this estimate rho(T) by power iteration instead\nPOWER_ITER_MAX = 200\nPOWER_ITER_TOL = 1e-6\n", "metrics": "# The single place a solve gets judged correct or not -- from the residual, not the solver's own claim.\nimport numpy as np\n\n# relative residual ||Ax - b|| / ||b|| below which a solve counts as successful\nSUCCESS_TOL = 1e-8\n\n# relative forward error, reported alongside but never used to gate success (ill-conditioning)\nACCURATE_TOL = 1e-6\n\nSTATUS_SOLVED = \"solved\"                    # residual verified below SUCCESS_TOL\nSTATUS_INACCURATE = \"inaccurate\"            # solver CLAIMED success; the residual says no\nSTATUS_NO_CONVERGE = \"did_not_converge\"     # solver admitted failure, residual agrees\nSTATUS_DIVERGED = \"diverged\"                # iterate blew up or went non-finite\nSTATUS_NOT_APPLICABLE = \"not_applicable\"    # method undefined on this matrix\nSTATUS_SINGULAR = \"structurally_singular\"   # matrix has no unique solution\nSTATUS_SKIPPED = \"skipped_too_large\"        # deliberately not attempted at this size\nSTATUS_ERROR = \"error\"                      # unexpected failure, see note\n\n# the method was defined on this matrix and got to try -- denominator for conditional success\nAPPLICABLE_STATUSES = frozenset({STATUS_SOLVED, STATUS_INACCURATE,\n                                 STATUS_NO_CONVERGE, STATUS_DIVERGED})\n\n\ndef score(A, x, b, x_true, b_norm=None, x_true_norm=None, reported_converged=None):\n    # Measure one solve and decide status; reported_converged is stored, never trusted.\n    b_norm = b_norm if b_norm else (np.linalg.norm(b) or 1.0)\n    x_true_norm = x_true_norm if x_true_norm else (np.linalg.norm(x_true) or 1.0)\n\n    if x is None or not np.all(np.isfinite(x)):\n        return {\"status\": STATUS_DIVERGED, \"residual_abs\": np.nan, \"error_abs\": np.nan,\n                \"residual_rel\": np.nan, \"error_rel\": np.nan,\n                \"reported_converged\": reported_converged, \"accurate\": False}\n\n    residual_abs = float(np.linalg.norm(A @ x - b))\n    error_abs = float(np.linalg.norm(x - x_true))\n    residual_rel = residual_abs / b_norm\n    error_rel = error_abs / x_true_norm\n\n    if residual_rel <= SUCCESS_TOL:\n        status = STATUS_SOLVED\n    elif reported_converged is False:\n        status = STATUS_NO_CONVERGE     # the solver said so itself, and it was right\n    else:\n        status = STATUS_INACCURATE      # solver claimed success (or stayed silent) and is wrong\n\n    return {\n        \"status\": status,\n        \"residual_abs\": residual_abs,\n        \"error_abs\": error_abs,\n        \"residual_rel\": residual_rel,\n        \"error_rel\": error_rel,\n        \"reported_converged\": reported_converged,\n        \"accurate\": bool(error_rel <= ACCURATE_TOL),\n    }\n\n\ndef blank(status, note=None):\n    # A metric row for a solve that never happened (skipped, singular, N/A).\n    row = {\"status\": status, \"residual_abs\": np.nan, \"error_abs\": np.nan,\n           \"residual_rel\": np.nan, \"error_rel\": np.nan,\n           \"reported_converged\": None, \"accurate\": False}\n    if note is not None:\n        row[\"note\"] = note\n    return row\n\n\ndef rates(df, method):\n    # Applicability and conditional success, kept separate -- never multiply them together.\n    rows = df[df[\"method\"] == method]\n    total = len(rows)\n    applicable = rows[\"status\"].isin(APPLICABLE_STATUSES).sum()\n    solved = (rows[\"status\"] == STATUS_SOLVED).sum()\n    return {\n        \"method\": method,\n        \"corpus\": total,\n        \"applicable\": int(applicable),\n        \"solved\": int(solved),\n        \"applicability\": applicable / total if total else float(\"nan\"),\n        \"conditional_success\": solved / applicable if applicable else float(\"nan\"),\n    }\n", "direct_solvers": "# Four direct solvers for Ax = b, from scratch on dense arrays with vectorized NumPy rows.\nimport numpy as np\n\n\nclass SolverNotApplicable(Exception):\n    pass  # matrix doesn't meet this method's requirements (e.g. Cholesky needs SPD)\n\n\ndef _partial_pivot(A: np.ndarray, b: np.ndarray, row: int) -> None:\n    # In-place: swap in the row below with the largest pivot-column magnitude.\n    n = A.shape[0]\n    pivot_row = row + int(np.argmax(np.abs(A[row:, row])))\n    if abs(A[pivot_row, row]) < 1e-14:\n        raise SolverNotApplicable(\"matrix is singular (zero pivot even after partial pivoting)\")\n    if pivot_row != row:\n        A[[row, pivot_row]] = A[[pivot_row, row]]\n        b[[row, pivot_row]] = b[[pivot_row, row]]\n\n\ndef gauss_elimination(A_dense: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, int]:\n    # Forward elimination with partial pivoting, then back-substitution.\n    n = A_dense.shape[0]\n    A = A_dense.copy()\n    b = b.copy()\n    steps = 0\n\n    for k in range(n - 1):\n        _partial_pivot(A, b, k)\n        factors = A[k + 1 :, k] / A[k, k]\n        A[k + 1 :, k:] -= np.outer(factors, A[k, k:])\n        b[k + 1 :] -= factors * b[k]\n        steps += 1\n\n    x = np.zeros(n)\n    for i in range(n - 1, -1, -1):\n        x[i] = (b[i] - A[i, i + 1 :] @ x[i + 1 :]) / A[i, i]\n    return x, steps\n\n\ndef gauss_jordan(A_dense: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, int]:\n    # Reduce all the way to the identity matrix, no separate back-substitution.\n    n = A_dense.shape[0]\n    A = A_dense.copy()\n    b = b.copy()\n    steps = 0\n\n    for k in range(n):\n        _partial_pivot(A, b, k)\n        pivot = A[k, k]\n        A[k, :] /= pivot\n        b[k] /= pivot\n        pivot_col = A[:, k].copy()\n        pivot_col[k] = 0.0\n        A -= np.outer(pivot_col, A[k, :])\n        b -= pivot_col * b[k]\n        steps += 1\n\n    return b, steps\n\n\ndef lu_solve(A_dense: np.ndarray, b: np.ndarray) -> tuple[np.ndarray, int]:\n    # Doolittle LU with partial pivoting, then forward then back substitution.\n    n = A_dense.shape[0]\n    U = A_dense.copy()\n    L = np.eye(n)\n    perm_b = b.copy()\n    steps = 0\n\n    for k in range(n - 1):\n        pivot_row = k + int(np.argmax(np.abs(U[k:, k])))\n        if abs(U[pivot_row, k]) < 1e-14:\n            raise SolverNotApplicable(\"matrix is singular (zero pivot even after partial pivoting)\")\n        if pivot_row != k:\n            U[[k, pivot_row]] = U[[pivot_row, k]]\n            perm_b[[k, pivot_row]] = perm_b[[pivot_row, k]]\n            if k > 0:\n                L[[k, pivot_row], :k] = L[[pivot_row, k], :k]\n        factors = U[k + 1 :, k] / U[k, k]\n        L[k + 1 :, k] = factors\n        U[k + 1 :, k:] -= np.outer(factors, U[k, k:])\n        steps += 1\n\n    # Forward substitution: L y = perm_b\n    y = np.zeros(n)\n    for i in range(n):\n        y[i] = perm_b[i] - L[i, :i] @ y[:i]\n\n    # Back substitution: U x = y\n    x = np.zeros(n)\n    for i in range(n - 1, -1, -1):\n        x[i] = (y[i] - U[i, i + 1 :] @ x[i + 1 :]) / U[i, i]\n    return x, steps\n\n\ndef cholesky_solve(A_dense: np.ndarray, b: np.ndarray, sym_tol: float = 1e-8) -> tuple[np.ndarray, int]:\n    # Cholesky (A = R^T R), valid only for symmetric positive-definite matrices.\n    n = A_dense.shape[0]\n    if not np.allclose(A_dense, A_dense.T, atol=sym_tol, rtol=sym_tol):\n        raise SolverNotApplicable(\"matrix is not symmetric\")\n\n    R = np.zeros((n, n))\n    steps = 0\n    for i in range(n):\n        diag_term = A_dense[i, i] - R[:i, i] @ R[:i, i]\n        if diag_term <= 0:\n            raise SolverNotApplicable(\"matrix is not positive-definite\")\n        R[i, i] = np.sqrt(diag_term)\n        if i + 1 < n:\n            R[i, i + 1 :] = (A_dense[i, i + 1 :] - R[:i, i] @ R[:i, i + 1 :]) / R[i, i]\n        steps += 1\n\n    # Solve R^T y = b (forward), then R x = y (back)\n    y = np.zeros(n)\n    for i in range(n):\n        y[i] = b[i] - R[:i, i] @ y[:i]\n        y[i] /= R[i, i]\n\n    x = np.zeros(n)\n    for i in range(n - 1, -1, -1):\n        x[i] = (y[i] - R[i, i + 1 :] @ x[i + 1 :]) / R[i, i]\n    return x, steps\n", "iterative_solvers": "# Iterative solvers for Ax = b; cost is tracked by Work, not wall-clock or iteration count.\nimport numpy as np\nimport scipy.sparse as sp\nimport scipy.sparse.linalg as spla\n\nfrom . import config\nfrom .direct_solvers import SolverNotApplicable\n\n\nclass Work:\n    # Counts the operations a solve actually performed -- the real cost metric here.\n\n    __slots__ = (\"matvecs\", \"tri_solves\", \"precond\", \"setup\")\n\n    def __init__(self):\n        self.matvecs = 0\n        self.tri_solves = 0\n        self.precond = 0\n        self.setup = 0.0      # seconds spent building a factorization, if any\n\n    def as_dict(self):\n        return {\"matvecs\": self.matvecs, \"tri_solves\": self.tri_solves,\n                \"precond_applies\": self.precond, \"setup_sec\": self.setup}\n\n\ndef _counting_operator(A, work):\n    # Wrap A so that every product a library solver takes gets counted.\n    def matvec(v):\n        work.matvecs += 1\n        return A @ v\n    return spla.LinearOperator(A.shape, matvec=matvec, dtype=np.float64)\n\n\ndef _counting_preconditioner(apply_fn, shape, work):\n    def matvec(v):\n        work.precond += 1\n        return apply_fn(v)\n    return spla.LinearOperator(shape, matvec=matvec, dtype=np.float64)\n\n\ndef _diagonal_or_raise(A, what):\n    d = A.diagonal()\n    if np.any(np.abs(d) < 1e-14):\n        raise SolverNotApplicable(f\"{what} needs a nonzero diagonal \"\n                                  f\"({int((np.abs(d) < 1e-14).sum())} zero entries)\")\n    return d\n\n\ndef _prefactor_lower(M):\n    # Factor a lower-triangular matrix once (no reordering/pivoting) for repeated solves.\n    return spla.splu(M.tocsc(), permc_spec=\"NATURAL\", diag_pivot_thresh=0.0)\n\n\n# --------------------------------------------------------------- stationary\n\n\ndef jacobi(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    # x <- x + D^-1 (b - Ax), one matvec per iteration.\n    work = Work()\n    d = _diagonal_or_raise(A, \"Jacobi\")\n    b_norm = np.linalg.norm(b) or 1.0\n    x = np.zeros(A.shape[0])\n    best = np.inf\n\n    for k in range(1, max_iter + 1):\n        r = b - A @ x\n        work.matvecs += 1\n        rr = np.linalg.norm(r) / b_norm\n        if not np.isfinite(rr):\n            return x, k, False, work\n        if rr <= tol:\n            return x, k, True, work\n        if rr > best * config.DIVERGENCE_GROWTH:\n            return x, k, False, work\n        best = min(best, rr)\n        x = x + r / d\n\n    return x, max_iter, False, work\n\n\ndef _sor_core(A, b, omega, max_iter, tol, label):\n    # Shared engine for Gauss-Seidel (omega=1) and SOR: (D+wL)delta=w*r, x<-x+delta.\n    work = Work()\n    _diagonal_or_raise(A, label)\n    n = A.shape[0]\n    b_norm = np.linalg.norm(b) or 1.0\n\n    M = (sp.tril(A, k=-1, format=\"csr\") * omega + sp.diags(A.diagonal())).tocsc()\n    try:\n        lu = _prefactor_lower(M)\n    except RuntimeError as e:                      # singular triangular part\n        raise SolverNotApplicable(f\"{label}: {e}\") from e\n\n    x = np.zeros(n)\n    best = np.inf\n\n    for k in range(1, max_iter + 1):\n        r = b - A @ x\n        work.matvecs += 1\n        rr = np.linalg.norm(r) / b_norm\n        if not np.isfinite(rr):\n            return x, k, False, work\n        if rr <= tol:\n            return x, k, True, work\n        if rr > best * config.DIVERGENCE_GROWTH:\n            return x, k, False, work\n        best = min(best, rr)\n        x = x + lu.solve(omega * r)\n        work.tri_solves += 1\n\n    return x, max_iter, False, work\n\n\ndef gauss_seidel(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    return _sor_core(A, b, 1.0, max_iter, tol, \"Gauss-Seidel\")\n\n\ndef sor(A, b, omega=config.SOR_OMEGA, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    return _sor_core(A, b, omega, max_iter, tol, \"SOR\")\n\ndef estimate_rho_jacobi(A, iters=None, seed=0, work=None):\n    # Power-iteration estimate of rho(T_J); matvecs charged to `work`, NaN if diagonal has a zero.\n    iters = config.POWER_ITERS_OMEGA if iters is None else iters\n    d = A.diagonal()\n    if np.any(np.abs(d) < 1e-14):\n        return float(\"nan\")\n    rng = np.random.default_rng(seed)\n    v = rng.normal(size=A.shape[0])\n    nv = np.linalg.norm(v)\n    if nv == 0:\n        return float(\"nan\")\n    v /= nv\n    lam = 0.0\n    for _ in range(iters):\n        w = v - (A @ v) / d\n        if work is not None:\n            work.matvecs += 1\n        nw = np.linalg.norm(w)\n        if nw < 1e-300:\n            return 0.0\n        lam = nw\n        v = w / nw\n    return float(lam)\n\n\ndef optimal_omega(rho):\n    # Young (1950) formula; falls back to 1.0 (plain Gauss-Seidel) when rho isn't usable.\n    if not np.isfinite(rho) or rho >= 1.0:\n        return 1.0\n    return float(np.clip(2.0 / (1.0 + np.sqrt(max(1.0 - rho * rho, 0.0))), 1.0, 1.95))\n\n\ndef sor_adaptive(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    # SOR with omega estimated per matrix instead of fixed at one value for the corpus.\n    work = Work()\n    rho = estimate_rho_jacobi(A, work=work)\n    omega = optimal_omega(rho)\n    x, iters, converged, w2 = _sor_core(A, b, omega, max_iter, tol, \"SOR (adaptive)\")\n    w2.matvecs += work.matvecs          # the estimate is part of this method's cost\n    return x, iters, converged, w2\n\n# --------------------------------------------------------------- Krylov\n\n\ndef _is_symmetric(A, tol=1e-8):\n    diff = abs(A - A.T)\n    return diff.nnz == 0 or diff.max() <= tol * max(abs(A).max(), 1.0)\n\n\ndef conjugate_gradient(A, b, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    # Unpreconditioned CG; needs symmetry, indefiniteness reported as p'Ap <= 0.\n    if not _is_symmetric(A):\n        raise SolverNotApplicable(\"Conjugate Gradient needs a symmetric matrix\")\n    work = Work()\n    b_norm = np.linalg.norm(b) or 1.0\n    x = np.zeros(A.shape[0])\n    r = b.copy()\n    p = r.copy()\n    rs = r @ r\n\n    if np.sqrt(rs) <= tol * b_norm:     # x=0 already solves it, else p'Ap=0 looks indefinite\n        return x, 0, True, work\n\n    for k in range(1, max_iter + 1):\n        Ap = A @ p\n        work.matvecs += 1\n        pAp = p @ Ap\n        if pAp <= 0:\n            raise SolverNotApplicable(\"Conjugate Gradient needs positive definiteness \"\n                                      f\"(p'Ap = {pAp:.3e} at iteration {k})\")\n        alpha = rs / pAp\n        x = x + alpha * p\n        r = r - alpha * Ap\n        rr = np.linalg.norm(r) / b_norm\n        if not np.isfinite(rr):\n            return x, k, False, work\n        if rr <= tol:\n            return x, k, True, work\n        rs_new = r @ r\n        p = r + (rs_new / rs) * p\n        rs = rs_new\n\n    return x, max_iter, False, work\n\n\ndef bicgstab(A, b, M=None, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    # BiCGSTAB via scipy, A and the preconditioner wrapped so their calls get counted.\n    work = Work()\n    op = _counting_operator(A, work)\n    pre = None if M is None else _counting_preconditioner(M, A.shape, work)\n    x, info = spla.bicgstab(op, b, rtol=tol, atol=0.0, maxiter=max_iter, M=pre)\n    return x, work.matvecs, info == 0, work\n\n\ndef pcg(A, b, M=None, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    work = Work()\n    op = _counting_operator(A, work)\n    pre = None if M is None else _counting_preconditioner(M, A.shape, work)\n    x, info = spla.cg(op, b, rtol=tol, atol=0.0, maxiter=max_iter, M=pre)\n    return x, work.matvecs, info == 0, work\n\n\ndef gmres(A, b, M=None, restart=30, max_iter=config.MAX_ITERATIONS, tol=config.TOLERANCE):\n    # Restarted GMRES, PETSc's default Krylov method.\n    work = Work()\n    op = _counting_operator(A, work)\n    pre = None if M is None else _counting_preconditioner(M, A.shape, work)\n    x, info = spla.gmres(op, b, rtol=tol, atol=0.0, restart=restart,\n                         maxiter=max_iter // restart or 1, M=pre)\n    return x, work.matvecs, info == 0, work\n\n\n# --------------------------------------------------------------- preconditioning\n\n\ndef build_ilu(A, work=None):\n    # Incomplete LU with a relax-and-retry ladder; returns an apply function or None.\n    import time\n    t0 = time.perf_counter()\n    Acsc = A.tocsc()\n    probe = np.random.default_rng(0).standard_normal(A.shape[0])\n    for drop, fill in config.ILU_LADDER:\n        try:\n            ilu = spla.spilu(Acsc, drop_tol=drop, fill_factor=fill)\n        except (RuntimeError, ValueError, MemoryError):\n            continue\n        try:\n            y = ilu.solve(probe)   # a factorization can still be unusable, tiny pivot -> inf\n        except Exception:\n            continue\n        if not np.all(np.isfinite(y)):\n            continue\n        if work is not None:\n            work.setup += time.perf_counter() - t0\n        return ilu.solve\n\n    d = A.diagonal()                          # last resort: diagonal scaling\n    if np.any(np.abs(d) < 1e-14):\n        return None\n    if work is not None:\n        work.setup += time.perf_counter() - t0\n    return lambda v: v / d\n", "reference_solvers": "# Library solvers (SuperLU, ILU+Krylov) -- what an engineer would actually reach for.\nimport time\n\nimport numpy as np\nimport scipy.sparse.linalg as spla\n\nfrom .direct_solvers import SolverNotApplicable\nfrom .iterative_solvers import Work, build_ilu, bicgstab, gmres, pcg, _is_symmetric\n\n\ndef sparse_lu(A, b):\n    # SuperLU factorization then a single triangular solve pair, counted as 1 iteration.\n    work = Work()\n    t0 = time.perf_counter()\n    try:\n        lu = spla.splu(A.tocsc())\n    except RuntimeError as e:\n        raise SolverNotApplicable(f\"SuperLU could not factor this matrix: {e}\") from e\n    work.setup = time.perf_counter() - t0\n    x = lu.solve(b)\n    work.tri_solves += 2\n    return x, 1, True, work\n\n\ndef sparse_spsolve(A, b):\n    # The one-line answer: ``spsolve(A, b)``. What a practitioner types.\n    work = Work()\n    t0 = time.perf_counter()\n    try:\n        x = spla.spsolve(A.tocsc(), b)\n    except RuntimeError as e:\n        raise SolverNotApplicable(f\"spsolve failed: {e}\") from e\n    work.setup = time.perf_counter() - t0\n    work.tri_solves += 2\n    return x, 1, True, work\n\n\ndef ilu_only(A, b):\n    # Zero-iteration control: apply the ILU factorization to b and stop, no Krylov on top.\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    x = apply_ilu(b)\n    work.precond += 1\n    return x, 0, True, work\n\n\ndef ilu_bicgstab(A, b):\n    # ILU-preconditioned BiCGSTAB, run unconditionally -- no dispatch.\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    x, its, conv, w = bicgstab(A, b, M=apply_ilu)\n    w.setup = work.setup\n    return x, its, conv, w\n\n\ndef ilu_gmres(A, b, restart=30):\n    # ILU-preconditioned restarted GMRES: PETSc's default configuration.\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    x, its, conv, w = gmres(A, b, M=apply_ilu, restart=restart)\n    w.setup = work.setup\n    return x, its, conv, w\n\n\ndef ilu_krylov_dispatched(A, b):\n    # PCG if A is symmetric else BiCGSTAB -- this is APK, kept as an honest baseline, not a novelty.\n    work = Work()\n    apply_ilu = build_ilu(A, work)\n    if apply_ilu is None:\n        raise SolverNotApplicable(\"no usable ILU factorization for this matrix\")\n    if _is_symmetric(A):\n        x, its, conv, w = pcg(A, b, M=apply_ilu)\n        if conv and np.all(np.isfinite(x)):\n            w.setup = work.setup\n            return x, its, conv, w\n    x, its, conv, w = bicgstab(A, b, M=apply_ilu)     # fallback and default path\n    w.setup = work.setup\n    return x, its, conv, w\n", "refinement": "# Iterative refinement (Wilkinson 1963), applied the same way to every solver.\nimport numpy as np\n\nfrom .iterative_solvers import Work\n\n\ndef refine(solver, A, b, passes=1):\n    # Run `solver` on (A, b), then apply `passes` correction steps, cost summed in.\n    x, iterations, converged, work = solver(A, b)\n    if passes <= 0 or x is None or not np.all(np.isfinite(x)):\n        return x, iterations, converged, work\n\n    total = Work()\n    total.matvecs = work.matvecs\n    total.tri_solves = work.tri_solves\n    total.precond = work.precond\n    total.setup = work.setup\n\n    b_norm = np.linalg.norm(b) or 1.0\n    for _ in range(passes):\n        r = b - A @ x\n        total.matvecs += 1\n        if not np.all(np.isfinite(r)):\n            break\n        # nothing left to correct -- a zero residual would make CG report a false failure\n        if np.linalg.norm(r) <= np.finfo(float).eps * b_norm:\n            break\n        d, its, conv, w = solver(A, r)\n        total.matvecs += w.matvecs\n        total.tri_solves += w.tri_solves\n        total.precond += w.precond\n        total.setup += w.setup\n        iterations += its\n        if d is None or not np.all(np.isfinite(d)):\n            break\n        x_next = x + d\n        if not np.all(np.isfinite(x_next)):\n            break\n        x = x_next\n        converged = converged or conv\n\n    return x, iterations, converged, total\n", "io_utils": "# Loading matrices and building the ground-truth (x_true, b) pair.\nfrom pathlib import Path\n\nimport numpy as np\nimport scipy.io\nimport scipy.sparse as sp\n\n\ndef load_matrix(mtx_path):\n    # Load a Matrix Market file as a real-valued square CSR matrix.\n    A = scipy.io.mmread(str(mtx_path))\n    A = sp.csr_matrix(A, dtype=np.float64)\n    if A.shape[0] != A.shape[1]:\n        raise ValueError(f\"{mtx_path} is not square: {A.shape}\")\n    return A\n\n\ndef make_ground_truth(A, seed=None):\n    # Return (x_true, b) with b = A @ x_true; seed draws x_true random instead of ones.\n    n = A.shape[0]\n    if seed is None:\n        x_true = np.ones(n, dtype=np.float64)\n    else:\n        x_true = np.random.default_rng(seed).standard_normal(n)\n    return x_true, A @ x_true\n\n\ndef cancellation_ratio(A, x_true):\n    # ||A x|| / |||A| |x|||: near machine epsilon, b is just rounding noise.\n    num = float(np.linalg.norm(A @ x_true))\n    den = float(np.linalg.norm(abs(A) @ np.abs(x_true)))\n    return num / den if den > 0 else np.nan\n\n\ndef structural_singularity(A):\n    # Count all-zero rows/columns -- no unique solution, and spilu OS-kills on these on Kaggle.\n    zero_rows = int((np.diff(A.indptr) == 0).sum())\n    zero_cols = int((np.diff(A.tocsc().indptr) == 0).sum())\n    return zero_rows, zero_cols\n\n\ndef structural_rank_deficit(A):\n    # n - structural_rank(A): stronger than structural_singularity, catches M10PI_n-style cases.\n    from scipy.sparse.csgraph import structural_rank\n    n = A.shape[0]\n    try:\n        return int(n - structural_rank(A.tocsr()))\n    except Exception:\n        return 0            # never let the screen itself be what fails the run\n\n\ndef discover_matrices(dataset_root):\n    # Find every .mtx under the corpus root, flat or nested, as one entry each.\n    root = Path(dataset_root)\n    entries = []\n    for domain_dir in sorted(p for p in root.iterdir() if p.is_dir()):\n        for mtx_path in sorted(domain_dir.rglob(\"*.mtx\")):\n            entries.append({\"domain\": domain_dir.name,\n                            \"name\": mtx_path.stem,\n                            \"path\": mtx_path})\n    return entries\n", "spectral": "# Spectral radii of the iteration matrices, and which classical hypothesis class applies.\nimport numpy as np\nimport scipy.sparse as sp\nimport scipy.sparse.linalg as spla\n\nfrom . import config\n\n\ndef _diagonal(A):\n    d = A.diagonal()\n    return None if np.any(np.abs(d) < 1e-14) else d\n\n\ndef jacobi_operator(A):\n    # T_J = I - D^-1 A, applied without forming it.\n    d = _diagonal(A)\n    if d is None:\n        return None\n    return spla.LinearOperator(A.shape, matvec=lambda v: v - (A @ v) / d, dtype=np.float64)\n\n\ndef gauss_seidel_operator(A):\n    # T_GS = -(D + L)^-1 U, with the triangular factor built once.\n    if _diagonal(A) is None:\n        return None\n    M = sp.tril(A, format=\"csc\")\n    U = sp.triu(A, k=1, format=\"csr\")\n    try:\n        lu = spla.splu(M, permc_spec=\"NATURAL\", diag_pivot_thresh=0.0)\n    except RuntimeError:\n        return None\n    return spla.LinearOperator(A.shape, matvec=lambda v: -lu.solve(U @ v), dtype=np.float64)\n\n\ndef _power_iteration(op, n, rng=None):\n    # Estimate the dominant eigenvalue magnitude by repeated application. Returns (rho, converged).\n    rng = rng or np.random.default_rng(0)\n    v = rng.standard_normal(n)\n    v /= np.linalg.norm(v) or 1.0\n    rho = np.nan\n    for _ in range(config.POWER_ITER_MAX):\n        w = op @ v\n        nw = np.linalg.norm(w)\n        if not np.isfinite(nw):\n            return np.inf, False\n        if nw == 0.0:\n            return 0.0, True\n        prev, rho = rho, nw\n        v = w / nw\n        if np.isfinite(prev) and abs(rho - prev) <= config.POWER_ITER_TOL * max(rho, 1e-30):\n            return float(rho), True\n    return float(rho), False\n\n\ndef dense_iteration_matrix(A, kind=\"jacobi\"):\n    # Build the iteration matrix densely, all columns at once rather than one by one.\n    d = _diagonal(A)\n    if d is None:\n        return None\n    n = A.shape[0]\n    if kind == \"jacobi\":\n        return np.eye(n) - A.toarray() / d[:, None]\n\n    M = sp.tril(A, format=\"csc\")\n    U = sp.triu(A, k=1, format=\"csr\")\n    try:\n        lu = spla.splu(M, permc_spec=\"NATURAL\", diag_pivot_thresh=0.0)\n    except RuntimeError:\n        return None\n    return -lu.solve(U.toarray())          # splu.solve takes all columns at once\n\n\ndef spectral_radius(A, kind=\"jacobi\", exact_cap=None):\n    # rho of an iteration matrix. Returns (rho, how), how = exact_eig/arpack/power_iteration.\n    exact_cap = config.SPECTRAL_EXACT_CAP if exact_cap is None else exact_cap\n    n = A.shape[0]\n    op = jacobi_operator(A) if kind == \"jacobi\" else gauss_seidel_operator(A)\n    if op is None:\n        return np.nan, \"not_applicable\"\n\n    if n <= exact_cap:\n        T = dense_iteration_matrix(A, kind)\n        try:\n            if T is not None:\n                return float(np.max(np.abs(np.linalg.eigvals(T)))), \"exact_eig\"\n        except (np.linalg.LinAlgError, MemoryError):\n            pass\n\n    try:\n        vals = spla.eigs(op, k=1, which=\"LM\", return_eigenvectors=False,\n                         maxiter=config.POWER_ITER_MAX * 10, tol=config.POWER_ITER_TOL)\n        return float(np.abs(vals[0])), \"arpack\"\n    except Exception:\n        rho, ok = _power_iteration(op, n)\n        return rho, \"power_iteration\" if ok else \"power_iteration_unconverged\"\n\n\ndef iterations_to_tolerance(rho, tol=None):\n    # How many iterations rho^k <= tol needs (asymptotic rate only); infinite when rho >= 1.\n    tol = config.TOLERANCE if tol is None else tol\n    if not np.isfinite(rho) or rho >= 1.0:\n        return np.inf\n    if rho <= 0.0:\n        return 1.0\n    return float(np.log(tol) / np.log(rho))\n\n\ndef convergence_verdict(rho, tol=None, budget=None):\n    # Four-way verdict: not_applicable (zero diagonal), diverges, too_slow, or converges.\n    budget = config.MAX_ITERATIONS if budget is None else budget\n    if rho is None or (isinstance(rho, float) and np.isnan(rho)):\n        return \"not_applicable\"\n    if not np.isfinite(rho) or rho >= 1.0:      # +inf is genuine divergence\n        return \"diverges\"\n    return \"converges\" if iterations_to_tolerance(rho, tol) <= budget else \"too_slow\"\n\n\ndef comparison_matrix(A):\n    # M(A): |a_ii| on the diagonal, -|a_ij| off it -- A is H-matrix iff M(A) is M-matrix.\n    M = -abs(A).tolil()\n    M.setdiag(np.abs(A.diagonal()))\n    return M.tocsr()\n\n\ndef is_strictly_diagonally_dominant(A):\n    d = np.abs(A.diagonal())\n    off = np.asarray(abs(A).sum(axis=1)).ravel() - d\n    return bool(np.all(d > off))\n\n\ndef is_weakly_diagonally_dominant(A):\n    d = np.abs(A.diagonal())\n    off = np.asarray(abs(A).sum(axis=1)).ravel() - d\n    return bool(np.all(d >= off) and np.any(d > off))\n\n\ndef is_l_matrix(A):\n    d = A.diagonal()\n    if np.any(d <= 0):\n        return False\n    off = A - sp.diags(d)\n    return bool(off.nnz == 0 or off.max() <= 0)\n\n\ndef is_symmetric(A, tol=1e-8):\n    diff = abs(A - A.T)\n    return bool(diff.nnz == 0 or diff.max() <= tol * max(abs(A).max(), 1.0))\n\n\ndef is_spd(A):\n    # Symmetric with a successful sparse Cholesky-equivalent factorization.\n    if not is_symmetric(A):\n        return False\n    try:\n        vals = spla.eigsh(A.astype(np.float64), k=1, which=\"SA\",\n                          return_eigenvectors=False, maxiter=5000)\n        return bool(vals[0] > 0)\n    except Exception:\n        try:\n            np.linalg.cholesky(A.toarray())\n            return True\n        except (np.linalg.LinAlgError, MemoryError, ValueError):\n            return False\n\n\ndef classify(A, rho_jacobi=None, rho_gs=None):\n    # Label a matrix with every hypothesis class it satisfies, plus one primary label.\n    flags = {\n        \"strict_diag_dominant\": is_strictly_diagonally_dominant(A),\n        \"weak_diag_dominant\": is_weakly_diagonally_dominant(A),\n        \"l_matrix\": is_l_matrix(A),\n        \"symmetric\": is_symmetric(A),\n        \"spd\": is_spd(A),\n    }\n\n    if rho_jacobi is None:\n        rho_jacobi, _ = spectral_radius(A, \"jacobi\")\n    if rho_gs is None:\n        rho_gs, _ = spectral_radius(A, \"gauss_seidel\")\n\n    rho_comparison, _ = spectral_radius(comparison_matrix(A), \"jacobi\")\n    flags[\"h_matrix\"] = bool(np.isfinite(rho_comparison) and rho_comparison < 1.0)\n\n    flags[\"m_matrix\"] = bool(flags[\"l_matrix\"] and np.isfinite(rho_jacobi) and rho_jacobi < 1.0)\n\n    # property A proxy: Young's rho(T_GS) = rho(T_J)^2 holding numerically\n    flags[\"property_a_proxy\"] = bool(\n        np.isfinite(rho_jacobi) and np.isfinite(rho_gs) and rho_jacobi > 0\n        and abs(rho_gs - rho_jacobi ** 2) <= 1e-3 * max(rho_jacobi ** 2, 1e-12)\n    )\n\n    for label in (\"m_matrix\", \"spd\", \"strict_diag_dominant\", \"h_matrix\",\n                  \"l_matrix\", \"weak_diag_dominant\", \"property_a_proxy\"):\n        if flags[label]:\n            primary = label\n            break\n    else:\n        primary = \"none\"\n\n    return {**flags,\n            \"rho_jacobi\": rho_jacobi,\n            \"rho_gauss_seidel\": rho_gs,\n            \"rho_comparison\": rho_comparison,\n            \"hypothesis_class\": primary,\n            \"jacobi_converges_predicted\": bool(np.isfinite(rho_jacobi) and rho_jacobi < 1.0),\n            \"gs_converges_predicted\": bool(np.isfinite(rho_gs) and rho_gs < 1.0),\n            \"jacobi_verdict\": convergence_verdict(rho_jacobi),\n            \"gs_verdict\": convergence_verdict(rho_gs),\n            \"jacobi_iterations_needed\": iterations_to_tolerance(rho_jacobi),\n            \"gs_iterations_needed\": iterations_to_tolerance(rho_gs),\n            \"optimal_omega\": (2.0 / (1.0 + np.sqrt(1.0 - rho_jacobi ** 2))\n                              if np.isfinite(rho_jacobi) and rho_jacobi < 1.0 else np.nan)}\n", "reordering": "# Row permutation to pick a better diagonal before Jacobi/Gauss-Seidel/SOR run.\nimport time\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\nOBJECTIVES = (\"bottleneck\", \"minsum\", \"mc64\", \"best\", \"none\")\n\n# \"best\" tries all of these and keeps whichever gives the lowest estimated rho(T_GS).\nPORTFOLIO = (\"none\", \"mc64\", \"minsum\", \"bottleneck\")\n\n# above this size fall back to sparse matching; dense is fine up to here on Kaggle's RAM.\nDENSE_ASSIGNMENT_CAP = 12000\n\n\ndef row_ratios(A):\n    # Row ratio each entry would give if picked as that row's diagonal.\n    M = abs(A).tocsr()\n    rowsum = np.asarray(M.sum(axis=1)).ravel()\n    R = M.copy().astype(np.float64)\n    counts = np.diff(R.indptr)\n    R.data = (np.repeat(rowsum, counts) - M.data) / M.data\n    return R\n\n\ndef worst_row_ratio(A):\n    # Worst row ratio for the diagonal A has now; below 1 = strict diagonal dominance.\n    M = abs(A).tocsr()\n    rowsum = np.asarray(M.sum(axis=1)).ravel()\n    d = np.abs(A.diagonal())\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        r = (rowsum - d) / d\n    r[d == 0] = np.inf\n    return float(np.max(r)) if r.size else np.inf\n\n\ndef _perm_from_matching(match_rows_to_cols, n):\n    p = np.empty(n, dtype=int)\n    p[match_rows_to_cols] = np.arange(n)\n    return p\n\n\ndef bottleneck_permutation(A):\n    # Minimise the worst row ratio, via binary search + bipartite matching.\n    n = A.shape[0]\n    if n > DENSE_ASSIGNMENT_CAP:\n        return None, np.inf\n\n    R = row_ratios(A)\n    if R.nnz == 0:\n        return None, np.inf\n\n    candidates = np.unique(R.data)\n\n    # MC64's own worst ratio upper-bounds the optimum, so anything above it can be dropped.\n    mc_perm, _ = mc64_permutation(A)\n    if mc_perm is not None:\n        upper = worst_row_ratio(A[mc_perm, :])\n        if np.isfinite(upper):\n            kept = candidates[candidates <= upper]\n            if kept.size:\n                candidates = kept\n\n    # cap the number of thresholds tried, quantiles beyond this many distinct values.\n    if candidates.size > 1024:\n        candidates = np.unique(np.quantile(candidates, np.linspace(0, 1, 1024)))\n\n    dense = R.toarray()\n    pattern = abs(A).toarray() > 0\n\n    def feasible(threshold):\n        # Perfect matching exists using only entries <= threshold?\n        allowed = pattern & (dense <= threshold)\n        cost = np.where(allowed, 0.0, 1.0)\n        rows, cols = linear_sum_assignment(cost)\n        if cost[rows, cols].sum() > 0:\n            return False, None\n        m = np.empty(n, dtype=int)\n        m[rows] = cols\n        return True, m\n\n    lo, hi, best = 0, candidates.size - 1, None\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        ok, m = feasible(candidates[mid])\n        if ok:\n            best = (np.asarray(m).copy(), candidates[mid])\n            hi = mid - 1\n        else:\n            lo = mid + 1\n    if best is None:\n        return None, np.inf\n    m, worst = best\n    return _perm_from_matching(m, n), float(worst)\n\n\ndef minsum_permutation(A):\n    # Minimise total row ratio -- turns out to be exactly MC64's own objective.\n    n = A.shape[0]\n    if n > DENSE_ASSIGNMENT_CAP:\n        return None, np.inf\n\n    R = row_ratios(A)\n    if R.nnz == 0:\n        return None, np.inf\n\n    # log1p so one huge-ratio row can't swamp the sum and make this collapse to bottleneck.\n    mask = abs(A).toarray() > 0\n    return _dense_assignment(np.log1p(R.toarray()), mask, n)\n\n\ndef _dense_assignment(weights, mask, n):\n    # Minimum-cost perfect assignment, dense (sparse matching hung on real inputs).\n    W = np.full((n, n), np.inf)\n    W[mask] = weights[mask]\n    finite = W[np.isfinite(W)]\n    if finite.size == 0:\n        return None, np.inf\n    penalty = (finite.max() + 1.0) * n + 1.0\n    W[~np.isfinite(W)] = penalty\n    rows, cols = linear_sum_assignment(W)\n    if np.any(W[rows, cols] >= penalty):\n        return None, np.inf\n    return _perm_from_matching(cols[np.argsort(rows)], n), float(W[rows, cols].sum())\n\n\ndef mc64_permutation(A):\n    # Baseline: maximise the product of |diagonal| entries, as MC64 does.\n    n = A.shape[0]\n    M = abs(A).tocsr().astype(np.float64)\n    if M.nnz == 0:\n        return None, np.inf\n\n    if n <= DENSE_ASSIGNMENT_CAP:\n        d = M.toarray()\n        mask = d > 0\n        with np.errstate(divide=\"ignore\"):\n            C = -np.log(d, out=np.full_like(d, -np.inf), where=mask)\n        C[mask] -= C[mask].min() - 1.0\n        return _dense_assignment(C, mask, n)\n\n    return None, np.inf\n\n\ndef apply_permutation(A, b, p):\n    # A' = PA, b' = Pb; the solution x is unchanged.\n    return A[p, :].tocsr(), b[p]\n\n\ndef select_diagonal(A, b=None, objective=\"bottleneck\"):\n    # Choose the diagonal, returning (A', b', info). objective=\"none\" is the control.\n    n = A.shape[0]\n    info = {\"objective\": objective, \"permuted\": False, \"cost\": np.nan,\n            \"zero_diagonal_before\": int((np.abs(A.diagonal()) < 1e-14).sum()),\n            \"worst_ratio_before\": worst_row_ratio(A)}\n\n    if objective == \"none\":\n        return A, b, info\n\n    if objective == \"best\":\n        return _select_best(A, b, info)\n\n    solver = {\"bottleneck\": bottleneck_permutation,\n              \"minsum\": minsum_permutation,\n              \"mc64\": mc64_permutation}[objective]\n    p, cost = solver(A)\n    if p is None:\n        info[\"note\"] = \"no perfect matching: no permutation can fill the diagonal\"\n        return A, b, info\n\n    A2 = A[p, :].tocsr()\n    b2 = None if b is None else b[p]\n    info.update(permuted=not np.array_equal(p, np.arange(n)), cost=cost,\n                zero_diagonal_after=int((np.abs(A2.diagonal()) < 1e-14).sum()),\n                worst_ratio_after=worst_row_ratio(A2))\n    return A2, b2, info\n\n\ndef _estimate_rho_gs(A, iters=40):\n    # Cheap power-iteration estimate of rho(T_GS), for ranking candidates only.\n    from . import spectral\n    op = spectral.gauss_seidel_operator(A)\n    if op is None:\n        return np.inf\n    v = np.random.default_rng(0).standard_normal(A.shape[0])\n    nv = np.linalg.norm(v)\n    if nv == 0:\n        return np.inf\n    v /= nv\n    rho = np.inf\n    for _ in range(iters):\n        w = op @ v\n        nw = np.linalg.norm(w)\n        if not np.isfinite(nw):\n            return np.inf\n        if nw == 0.0:\n            return 0.0\n        rho, v = nw, w / nw\n    return float(rho)\n\n\ndef _select_best(A, b, info):\n    # Try every candidate, keep whichever gives the smallest estimated rho(T_GS).\n    best = (np.inf, None, None, \"none\")\n    tried, ratios = {}, {}\n    for name in PORTFOLIO:\n        if name == \"none\":\n            cand = A\n        else:\n            p, _ = {\"mc64\": mc64_permutation, \"minsum\": minsum_permutation,\n                    \"bottleneck\": bottleneck_permutation}[name](A)\n            if p is None:\n                continue\n            cand = A[p, :].tocsr()\n        ratios[name] = worst_row_ratio(cand)\n        if np.any(np.abs(cand.diagonal()) < 1e-14):\n            tried[name] = np.inf\n            continue\n        rho = _estimate_rho_gs(cand)\n        tried[name] = rho\n        if rho < best[0]:\n            best = (rho, cand, None if name == \"none\" else p, name)\n\n    rho, cand, p, name = best\n    info.update(chosen=name, estimated_rho_gs=rho, candidates=tried, ratios=ratios,\n                permuted=name != \"none\")\n    if cand is None:\n        info[\"note\"] = \"no candidate produced a usable diagonal\"\n        return A, b, info\n    info[\"zero_diagonal_after\"] = 0\n    info[\"worst_ratio_after\"] = worst_row_ratio(cand)\n    return cand, (b if p is None or b is None else b[p]), info\n\n\ndef make_solver(base_solver, objective=\"bottleneck\"):\n    # Wrap a stationary solver so it selects its diagonal first.\n    def solve(A, b):\n        t0 = time.perf_counter()\n        A2, b2, _ = select_diagonal(A, b, objective=objective)\n        setup = time.perf_counter() - t0\n        x, iters, converged, work = base_solver(A2, b2)\n        # count the permutation as this method's own setup cost, not a free head start.\n        if work is not None and hasattr(work, \"setup\"):\n            work.setup += setup\n        return x, iters, converged, work\n    solve.__name__ = f\"{base_solver.__name__}_{objective}\"\n    return solve\n", "benchmark": "# The benchmark harness: run every method on every matrix, score from the residual only.\nimport time\nimport warnings\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nimport scipy.sparse.linalg as spla\n\nfrom . import config, direct_solvers as direct, iterative_solvers as it\nfrom . import reordering\nfrom . import metrics, reference_solvers as ref, spectral\nfrom .direct_solvers import SolverNotApplicable\nfrom .io_utils import (cancellation_ratio, discover_matrices, load_matrix,\n                       make_ground_truth, structural_singularity,\n                       structural_rank_deficit)\nfrom .refinement import refine\n\n\nclass Method:\n    # One benchmarked method and the facts the harness needs about it.\n\n    def __init__(self, name, fn, family, dense=False, capped=False):\n        self.name = name\n        self.fn = fn\n        self.family = family\n        self.dense = dense      # takes a dense array rather than a sparse matrix\n        self.capped = capped    # subject to DIRECT_SIZE_CAP\n\n    def __repr__(self):\n        return f\"<Method {self.name}>\"\n\n\ndef _wrap_dense(fn):\n    # Adapt a hand-written direct solver to the common 4-tuple contract.\n    def call(A_dense, b):\n        x, steps = fn(A_dense, b)\n        return x, steps, None, it.Work()      # None: it makes no convergence claim\n    return call\n\n\n# hand-written direct solvers are pure-python O(n^3), ~50-100x slower than library calls\nMETHODS = [\n    Method(\"Gauss elimination\", _wrap_dense(direct.gauss_elimination), \"direct-handwritten\", dense=True, capped=True),\n    Method(\"Gauss-Jordan\", _wrap_dense(direct.gauss_jordan), \"direct-handwritten\", dense=True, capped=True),\n    Method(\"LU\", _wrap_dense(direct.lu_solve), \"direct-handwritten\", dense=True, capped=True),\n    Method(\"Cholesky\", _wrap_dense(direct.cholesky_solve), \"direct-handwritten\", dense=True, capped=True),\n\n    Method(\"spsolve (SuperLU)\", ref.sparse_spsolve, \"direct-library\"),\n    Method(\"splu (SuperLU)\", ref.sparse_lu, \"direct-library\"),\n\n    Method(\"Jacobi\", it.jacobi, \"stationary\"),\n    Method(\"Gauss-Seidel\", it.gauss_seidel, \"stationary\"),\n    Method(\"SOR\", it.sor, \"stationary\"),\n\n    Method(\"Conjugate Gradient\", it.conjugate_gradient, \"krylov\"),\n    Method(\"BiCGSTAB\", it.bicgstab, \"krylov\"),\n    Method(\"GMRES(30)\", lambda A, b: it.gmres(A, b, restart=30), \"krylov\"),\n\n    Method(\"ILU only\", ref.ilu_only, \"preconditioned\"),\n    Method(\"ILU-BiCGSTAB\", ref.ilu_bicgstab, \"preconditioned\"),\n    Method(\"ILU-GMRES(30)\", ref.ilu_gmres, \"preconditioned\"),\n    Method(\"ILU-Krylov (dispatched)\", ref.ilu_krylov_dispatched, \"preconditioned\"),\n]\n\n# reordering + adaptive omega in front of unmodified solvers, each arm measured separately\nPIPELINE_METHODS = [\n    Method(\"SOR (adaptive w)\", it.sor_adaptive, \"pipeline\"),\n\n    Method(\"Jacobi + reordered\", reordering.make_solver(it.jacobi, \"best\"), \"pipeline\"),\n    Method(\"Gauss-Seidel + reordered\",\n           reordering.make_solver(it.gauss_seidel, \"best\"), \"pipeline\"),\n    Method(\"SOR + reordered\", reordering.make_solver(it.sor, \"best\"), \"pipeline\"),\n\n    Method(\"SOR + pipeline\", reordering.make_solver(it.sor_adaptive, \"best\"), \"pipeline\"),\n\n    Method(\"ILU-BiCGSTAB + reordered\",\n           reordering.make_solver(ref.ilu_bicgstab, \"best\"), \"pipeline\"),\n    Method(\"ILU-Krylov + reordered\",\n           reordering.make_solver(ref.ilu_krylov_dispatched, \"best\"), \"pipeline\"),\n]\n\nPIPELINE_NAMES = [m.name for m in PIPELINE_METHODS]\n\nMETHOD_NAMES = [m.name for m in METHODS]\n\n\ndef condition_number(A, A_dense, n):\n    # Exact via SVD where affordable, a 1-norm estimate otherwise; route returned too.\n    with warnings.catch_warnings():\n        warnings.simplefilter(\"ignore\")\n        if A_dense is not None and n <= config.COND_EXACT_CAP:\n            try:\n                return float(np.linalg.cond(A_dense)), \"exact_svd\"\n            except (np.linalg.LinAlgError, MemoryError):\n                pass\n        try:\n            lu = spla.splu(A.tocsc())\n            inv_norm = spla.onenormest(spla.LinearOperator(\n                A.shape, matvec=lu.solve, rmatvec=lambda v: lu.solve(v, \"T\")))\n            return float(spla.onenormest(A) * inv_norm), \"estimate_1norm\"\n        except Exception:\n            return float(\"inf\"), \"failed\"\n\n\ndef _row(base, method, passes, **extra):\n    return {**base, \"method\": method.name, \"family\": method.family,\n            \"refinement_passes\": passes, **extra}\n\n\ndef run_one_matrix(domain, name, mtx_path, refinement_passes=(0, 1),\n                   with_spectral=True, seed=None, verbose=True, methods=None):\n    # Benchmark a set of methods on one matrix. Returns (result_rows, spectral_row).\n    methods = METHODS if methods is None else methods\n    try:\n        A = load_matrix(mtx_path)\n    except Exception as e:\n        base = {\"domain\": domain, \"matrix\": name, \"n\": np.nan, \"nnz\": np.nan}\n        rows = [_row(base, m, p, **metrics.blank(\"load_failed\", f\"{type(e).__name__}: {e}\"))\n                for m in methods for p in refinement_passes]\n        return rows, {\"domain\": domain, \"matrix\": name, \"status\": \"load_failed\"}\n\n    n = A.shape[0]\n    base = {\"domain\": domain, \"matrix\": name, \"n\": n, \"nnz\": A.nnz,\n            \"density\": A.nnz / (n * n)}\n\n    zero_rows, zero_cols = structural_singularity(A)\n    deficit = structural_rank_deficit(A)\n    if zero_rows or zero_cols or deficit:\n        if verbose:\n            print(f\"    structurally singular ({zero_rows} zero rows, {zero_cols} zero \"\n                  f\"cols, structural rank deficit {deficit}) -- no unique solution, \"\n                  \"all methods skipped\")\n        note = (f\"{zero_rows} zero rows, {zero_cols} zero cols, \"\n                f\"structural rank deficit {deficit}\")\n        rows = [_row(base, m, p, **metrics.blank(metrics.STATUS_SINGULAR, note))\n                for m in methods for p in refinement_passes]\n        return rows, {**base, \"status\": metrics.STATUS_SINGULAR}\n\n    x_true, b = make_ground_truth(A, seed=seed)\n    b_norm = np.linalg.norm(b) or 1.0\n    xt_norm = np.linalg.norm(x_true) or 1.0\n\n    needs_dense = n <= max(config.DIRECT_SIZE_CAP, config.COND_EXACT_CAP)\n    A_dense = A.toarray() if needs_dense else None\n\n    cond, cond_how = condition_number(A, A_dense, n)\n    base.update(condition_number=cond, condition_method=cond_how,\n                ill_conditioned=bool(cond > config.ILL_CONDITIONED),\n                cancellation_ratio=cancellation_ratio(A, x_true))\n    if verbose:\n        print(f\"    n={n:,} nnz={A.nnz:,} cond={cond:.2e} ({cond_how})\")\n\n    rows = []\n    for m in methods:\n        operand = A_dense if m.dense else A\n        for passes in refinement_passes:\n            if m.capped and n > config.DIRECT_SIZE_CAP:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_SKIPPED,\n                                                 f\"n={n} > cap {config.DIRECT_SIZE_CAP}\")))\n                continue\n            if m.dense and A_dense is None:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_SKIPPED, \"dense form not built\")))\n                continue\n            try:\n                t0 = time.perf_counter()\n                if passes:\n                    x, iters, conv, work = refine(m.fn, operand, b, passes=passes)\n                else:\n                    x, iters, conv, work = m.fn(operand, b)\n                elapsed = time.perf_counter() - t0\n                scored = metrics.score(A, x, b, x_true, b_norm, xt_norm, conv)\n                rows.append(_row(base, m, passes, runtime_sec=elapsed, iterations=iters,\n                                 **work.as_dict(), **scored))\n            except SolverNotApplicable as e:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_NOT_APPLICABLE, str(e))))\n            except (MemoryError, RuntimeError, ValueError, ZeroDivisionError) as e:\n                rows.append(_row(base, m, passes,\n                                 **metrics.blank(metrics.STATUS_ERROR, f\"{type(e).__name__}: {e}\")))\n        if verbose:\n            last = rows[-1]\n            print(f\"      {m.name:<24s} {last['status']}\")\n\n    spec = {**base, \"status\": \"analysed\"}\n    if with_spectral:\n        try:\n            rho_j, how_j = spectral.spectral_radius(A, \"jacobi\")\n            rho_g, how_g = spectral.spectral_radius(A, \"gauss_seidel\")\n            spec.update(spectral.classify(A, rho_j, rho_g),\n                        rho_jacobi_method=how_j, rho_gs_method=how_g)\n        except (MemoryError, ValueError) as e:\n            spec.update(status=f\"spectral_failed: {type(e).__name__}\")\n\n    return rows, spec\n\n\ndef run_full_benchmark(dataset_root, results_csv, spectral_csv=None,\n                       refinement_passes=(0, 1), with_spectral=True,\n                       seed=None, verbose=True):\n    # Sweep the whole corpus, checkpointing after every matrix.\n    entries = discover_matrices(dataset_root)\n    all_rows, all_spec = [], []\n    t0 = time.perf_counter()\n\n    for i, entry in enumerate(entries, 1):\n        if verbose:\n            print(f\"\\n[{i}/{len(entries)}] {entry['domain']}/{entry['name']}\"\n                  f\"   ({(time.perf_counter() - t0) / 60:.1f} min elapsed)\")\n        rows, spec = run_one_matrix(entry[\"domain\"], entry[\"name\"], entry[\"path\"],\n                                    refinement_passes=refinement_passes,\n                                    with_spectral=with_spectral, seed=seed, verbose=verbose)\n        all_rows.extend(rows)\n        all_spec.append(spec)\n        pd.DataFrame(all_rows).to_csv(results_csv, index=False)     # checkpoint\n        if spectral_csv:\n            pd.DataFrame(all_spec).to_csv(spectral_csv, index=False)\n\n    results = pd.DataFrame(all_rows)\n    if verbose:\n        expected = len(entries) * len(METHODS) * len(refinement_passes)\n        print(f\"\\nSweep complete in {(time.perf_counter() - t0) / 60:.1f} min\")\n        print(f\"  rows {len(results):,} of {expected:,} expected\"\n              f\"  |  matrices {results['matrix'].nunique()} of {len(entries)}\")\n        print(results[\"status\"].value_counts().to_string())\n    return results, pd.DataFrame(all_spec)\n\n\ndef summarise(results, passes=0):\n    # Applicability and conditional success per method -- kept separate, never multiplied.\n    subset = results[results[\"refinement_passes\"] == passes]\n    return pd.DataFrame([metrics.rates(subset, m) for m in METHOD_NAMES\n                         if m in set(subset[\"method\"])])\n", "__init__": "# Direct and iterative linear solvers benchmarked on real sparse matrices.\nfrom . import (benchmark, config, direct_solvers, io_utils, iterative_solvers,\n               metrics, reference_solvers, refinement, spectral)\nfrom .benchmark import METHODS, METHOD_NAMES, run_full_benchmark, run_one_matrix, summarise\nfrom .direct_solvers import SolverNotApplicable\nfrom .io_utils import discover_matrices, load_matrix, make_ground_truth\nfrom .metrics import score\nfrom .refinement import refine\n\n__all__ = [\n    \"benchmark\", \"config\", \"direct_solvers\", \"io_utils\", \"iterative_solvers\",\n    \"metrics\", \"reference_solvers\", \"refinement\", \"spectral\",\n    \"METHODS\", \"METHOD_NAMES\", \"run_full_benchmark\", \"run_one_matrix\", \"summarise\",\n    \"SolverNotApplicable\", \"discover_matrices\", \"load_matrix\", \"make_ground_truth\",\n    \"score\", \"refine\",\n]\n"}""")

_pkg = pathlib.Path("/kaggle/working/solvebench")
if not pathlib.Path("/kaggle").exists():
    _pkg = pathlib.Path("solvebench")
_pkg.mkdir(parents=True, exist_ok=True)
for _name, _src in _SOURCES.items():
    (_pkg / f"{_name}.py").write_text(_src, encoding="utf-8")
sys.path.insert(0, str(_pkg.parent))

import solvebench
from solvebench import benchmark, config, io_utils, metrics, reordering, spectral
print(f"library loaded: {len(_SOURCES)} modules, "
      f"{len(benchmark.METHODS)} methods")
for _m in benchmark.METHODS:
    print(f"   {_m.family:<20s} {_m.name}")

library loaded: 10 modules, 16 methods
   direct-handwritten   Gauss elimination
   direct-handwritten   Gauss-Jordan
   direct-handwritten   LU
   direct-handwritten   Cholesky
   direct-library       spsolve (SuperLU)
   direct-library       splu (SuperLU)
   stationary           Jacobi
   stationary           Gauss-Seidel
   stationary           SOR
   krylov               Conjugate Gradient
   krylov               BiCGSTAB
   krylov               GMRES(30)
   preconditioned       ILU only
   preconditioned       ILU-BiCGSTAB
   preconditioned       ILU-GMRES(30)
   preconditioned       ILU-Krylov (dispatched)


In [3]:
import platform, numpy, scipy, pandas, multiprocessing, json, time
from datetime import datetime

ON_KAGGLE = pathlib.Path("/kaggle").exists()
ENV = {
    "started": datetime.now().isoformat(),
    "on_kaggle": ON_KAGGLE,
    "python": platform.python_version(),
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "pandas": pandas.__version__,
    "cpu_count": multiprocessing.cpu_count(),
    "platform": platform.platform(),
}
for k, v in ENV.items():
    print(f"  {k:<12s} {v}")

# scipy 1.16.3 (Kaggle) vs 1.17.x (local): spilu OS-kills on 1.16.3 instead of raising

  started      2026-09-09T08:23:35.387862
  on_kaggle    True
  python       3.12.13
  numpy        2.0.2
  scipy        1.16.3
  pandas       2.3.3
  cpu_count    4
  platform     Linux-6.12.90+-x86_64-with-glibc2.35


In [4]:
DATA_ROOT = None
for _candidate in (pathlib.Path("/kaggle/input"), pathlib.Path("dataset_large")):
    if not _candidate.exists():
        continue
    _hits = list(_candidate.rglob("*.mtx"))
    if _hits:
        # Walk up from any matrix to the directory holding the domain folders.
        DATA_ROOT = _hits[0].parent.parent
        break

if DATA_ROOT is None:
    # a just-attached dataset can take a moment to propagate, hence this check
    print("NO MATRICES FOUND. Contents of /kaggle/input:")
    for _p in sorted(pathlib.Path("/kaggle/input").rglob("*"))[:40]:
        print("   ", _p)
    raise FileNotFoundError("dataset not attached or not yet propagated")

MATRICES = io_utils.discover_matrices(DATA_ROOT)
print(f"corpus root : {DATA_ROOT}")
print(f"matrices    : {len(MATRICES)}")
print(f"domains     : {len({m['domain'] for m in MATRICES})}")

# off Kaggle this only runs under test_notebooks.py, so keep its output away from real results
OUT_DIR = pathlib.Path("/kaggle/working/output" if ON_KAGGLE else ".nbscratch/output")
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
print(f"output      : {OUT_DIR}")

corpus root : /kaggle/input/datasets/mdmarufhasanrubab/solvebench-matrices-large
matrices    : 930
domains     : 27
output      : /kaggle/working/output


In [5]:
T0 = time.perf_counter()
import pandas as pd

# refinement disabled for everyone here -- measured separately in the refinement-study notebook
rows, specs = [], []
results_csv = OUT_DIR / "tables" / "benchmark_results.csv"

order = sorted(MATRICES, key=lambda e: e["path"].stat().st_size)
for i, entry in enumerate(order, 1):
    print(f"\n[{i}/{len(order)}] {entry['domain']}/{entry['name']}"
          f"   ({(time.perf_counter()-T0)/60:.1f} min)")
    r, _ = benchmark.run_one_matrix(entry["domain"], entry["name"], entry["path"],
                                    refinement_passes=(0,), with_spectral=False,
                                    verbose=True)
    rows.extend(r)
    pd.DataFrame(rows).to_csv(results_csv, index=False)   # checkpoint every matrix

results = pd.DataFrame(rows)
print(f"\nrows {len(results):,}  matrices {results['matrix'].nunique()}")
print(results["status"].value_counts().to_string())


[1/930] random_matrices/rand_001_n5   (0.0 min)
    n=5 nnz=25 cond=6.84e+01 (exact_svd)
      Gauss elimination        solved
      Gauss-Jordan             solved
      LU                       solved
      Cholesky                 not_applicable
      spsolve (SuperLU)        solved
      splu (SuperLU)           solved
      Jacobi                   did_not_converge
      Gauss-Seidel             did_not_converge
      SOR                      did_not_converge
      Conjugate Gradient       not_applicable
      BiCGSTAB                 solved
      GMRES(30)                solved
      ILU only                 solved
      ILU-BiCGSTAB             solved
      ILU-GMRES(30)            solved
      ILU-Krylov (dispatched)  solved

[2/930] chemical_process_simulation_problem/b1_ss   (0.0 min)
    n=7 nnz=15 cond=1.97e+02 (exact_svd)
      Gauss elimination        solved
      Gauss-Jordan             solved
      LU                       solved
      Cholesky                 not_app

/kaggle/working/solvebench/reference_solvers.py:48: MatrixRankWarning: Matrix is exactly singular
  x = spla.spsolve(A.tocsc(), b)


      GMRES(30)                did_not_converge
      ILU only                 inaccurate
      ILU-BiCGSTAB             solved
      ILU-GMRES(30)            solved
      ILU-Krylov (dispatched)  solved

[15/930] random_matrices/rand_006_n10   (0.1 min)
    n=10 nnz=100 cond=2.62e+02 (exact_svd)
      Gauss elimination        solved
      Gauss-Jordan             solved
      LU                       solved
      Cholesky                 not_applicable
      spsolve (SuperLU)        solved
      splu (SuperLU)           solved
      Jacobi                   did_not_converge
      Gauss-Seidel             did_not_converge
      SOR                      did_not_converge
      Conjugate Gradient       not_applicable
      BiCGSTAB                 solved
      GMRES(30)                solved
      ILU only                 solved
      ILU-BiCGSTAB             solved
      ILU-GMRES(30)            solved
      ILU-Krylov (dispatched)  solved

[16/930] structural_problem/bcsstm04   (0.1 min

/kaggle/working/solvebench/reference_solvers.py:48: MatrixRankWarning: Matrix is exactly singular
  x = spla.spsolve(A.tocsc(), b)


      BiCGSTAB                 inaccurate
      GMRES(30)                did_not_converge
      ILU only                 inaccurate
      ILU-BiCGSTAB             solved
      ILU-GMRES(30)            solved
      ILU-Krylov (dispatched)  solved

[20/930] undirected_weighted_graph/GD97_b   (0.2 min)
    structurally singular (1 zero rows, 1 zero cols) -- no unique solution, all methods skipped

[21/930] structural_problem/mesh1em6   (0.2 min)
    n=48 nnz=306 cond=6.11e+00 (exact_svd)
      Gauss elimination        solved
      Gauss-Jordan             solved
      LU                       solved
      Cholesky                 solved
      spsolve (SuperLU)        solved
      splu (SuperLU)           solved
      Jacobi                   solved
      Gauss-Seidel             solved
      SOR                      solved
      Conjugate Gradient       solved
      BiCGSTAB                 solved
      GMRES(30)                solved
      ILU only                 inaccurate
      ILU-Bi

...

      ILU-Krylov (dispatched)  did_not_converge

[726/930] circuit_simulation_problem/rajat03   (170.2 min)
    n=7,602 nnz=32,653 cond=2.48e+07 (estimate_1norm)
      Gauss elimination        skipped_too_large
      Gauss-Jordan             skipped_too_large
      LU                       skipped_too_large
      Cholesky                 skipped_too_large
      spsolve (SuperLU)        solved
      splu (SuperLU)           solved
      Jacobi                   not_applicable
      Gauss-Seidel             not_applicable
      SOR                      not_applicable
      Conjugate Gradient       not_applicable
      BiCGSTAB                 solved
      GMRES(30)                did_not_converge
      ILU only                 inaccurate
      ILU-BiCGSTAB             solved
      ILU-GMRES(30)            solved
      ILU-Krylov (dispatched)  solved

[727/930] optimization_problem/c-30   (170.3 min)
    n=5,321 nnz=65,693 cond=2.10e+13 (estimate_1norm)
      Gauss elimination        skip

/kaggle/working/solvebench/reference_solvers.py:48: MatrixRankWarning: Matrix is exactly singular
  x = spla.spsolve(A.tocsc(), b)


      BiCGSTAB                 did_not_converge
      GMRES(30)                did_not_converge
      ILU only                 not_applicable
      ILU-BiCGSTAB             not_applicable
      ILU-GMRES(30)            not_applicable
      ILU-Krylov (dispatched)  not_applicable

[778/930] structural_problem/msc04515   (188.0 min)
    n=4,515 nnz=97,707 cond=5.48e+06 (estimate_1norm)
      Gauss elimination        skipped_too_large
      Gauss-Jordan             skipped_too_large
      LU                       skipped_too_large
      Cholesky                 skipped_too_large
      spsolve (SuperLU)        solved
      splu (SuperLU)           solved
      Jacobi                   did_not_converge
      Gauss-Seidel             did_not_converge
      SOR                      did_not_converge
      Conjugate Gradient       solved
      BiCGSTAB                 solved
      GMRES(30)                did_not_converge
      ILU only                 inaccurate
      ILU-BiCGSTAB             

/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_isolve/iterative.py:408: RuntimeWarning: invalid value encountered in scalar divide
  beta = rho_cur / rho_prev


      ILU-Krylov (dispatched)  did_not_converge

[878/930] computational_fluid_dynamics_problem/cavity16   (255.2 min)
    n=4,562 nnz=138,187 cond=1.39e+07 (estimate_1norm)
      Gauss elimination        skipped_too_large
      Gauss-Jordan             skipped_too_large
      LU                       skipped_too_large
      Cholesky                 skipped_too_large
      spsolve (SuperLU)        solved
      splu (SuperLU)           solved
      Jacobi                   not_applicable
      Gauss-Seidel             not_applicable
      SOR                      not_applicable
      Conjugate Gradient       not_applicable
      BiCGSTAB                 solved
      GMRES(30)                did_not_converge
      ILU only                 inaccurate
      ILU-BiCGSTAB             did_not_converge
      ILU-GMRES(30)            did_not_converge
      ILU-Krylov (dispatched)  did_not_converge

[879/930] structural_problem/s3rmq4m1   (256.3 min)
    n=5,489 nnz=281,111 cond=3.06e+10 (estima

In [6]:
summary = benchmark.summarise(results, passes=0)
summary.to_csv(OUT_DIR / "tables" / "method_summary.csv", index=False)
print("Applicability and conditional success are separate columns and are never")
print("multiplied: doing that reported Conjugate Gradient at 10.4% when its")
print("conditional success is 83.9%.\n")
print(summary.to_string(index=False))

# Per-domain, with the denominator carried so a rate can always be checked.
per_domain = (results[results.refinement_passes == 0]
              .groupby(["domain", "method"])
              .agg(n=("status", "size"),
                   applicable=("status", lambda s: s.isin(metrics.APPLICABLE_STATUSES).sum()),
                   solved=("status", lambda s: (s == metrics.STATUS_SOLVED).sum()))
              .reset_index())
per_domain.to_csv(OUT_DIR / "tables" / "per_domain.csv", index=False)
print(f"\nper-domain rows: {len(per_domain)}")

Applicability and conditional success are separate columns and are never
multiplied: doing that reported Conjugate Gradient at 10.4% when its
conditional success is 83.9%.

                 method  corpus  applicable  solved  applicability  conditional_success
      Gauss elimination     930         598     595       0.643011             0.994983
           Gauss-Jordan     930         596     539       0.640860             0.904362
                     LU     930         598     595       0.643011             0.994983
               Cholesky     930          62      60       0.066667             0.967742
      spsolve (SuperLU)     930         904     876       0.972043             0.969027
         splu (SuperLU)     930         888     876       0.954839             0.986486
                 Jacobi     930         414      75       0.445161             0.181159
           Gauss-Seidel     930         414     121       0.445161             0.292271
                    SOR     930    

In [7]:
ENV["finished"] = datetime.now().isoformat()
ENV["runtime_minutes"] = (time.perf_counter() - T0) / 60
ENV["notebook"] = "solvebench-main-sweep"
ENV["config"] = {k: getattr(config, k) for k in dir(config)
                 if k.isupper() and not k.startswith("_")}
(OUT_DIR / "logs" / "run_metadata.json").write_text(json.dumps(ENV, indent=2, default=str))

print("=" * 70)
print(f"solvebench-main-sweep COMPLETE in {ENV['runtime_minutes']:.1f} min")
print("=" * 70)
for _t in ["benchmark_results.csv", "method_summary.csv", "per_domain.csv"]:
    _f = OUT_DIR / "tables" / _t
    print(f"  {_t:<34s} {_f.stat().st_size/1024:>9.1f} KB" if _f.exists()
          else f"  {_t:<34s} MISSING")

solvebench-main-sweep COMPLETE in 311.3 min
  benchmark_results.csv                 4022.0 KB
  method_summary.csv                       1.0 KB
  per_domain.csv                          19.1 KB
